# tk-assistant · Экспорт GGUF (adapter_e3 → Q4_K_M)

**Runtime → T4 GPU.** На выходе — файл `*.gguf` (~4.4 ГБ) для деплоя на CPU.

Адаптер берётся из той же сессии, где шло обучение (`outputs/adapter_e3_lr2e-4`).
Если сессия перезапускалась — ячейка 2 предложит загрузить `adapter_e3_lr2e-4.zip`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%capture
!pip install unsloth

## 1. Достаём адаптер (из сессии или из zip)

In [ ]:
import os, pathlib

ADAPTER = "outputs/adapter_e3_lr2e-4"
has_adapter = (pathlib.Path(ADAPTER) / "adapter_config.json").exists()

if not has_adapter:
    print("Адаптера в сессии нет — загрузи adapter_e3_lr2e-4.zip из панели файлов ниже:")
    from google.colab import files
    up = files.upload()  # выбери adapter_e3_lr2e-4.zip
    name = next(iter(up))
    os.system(f"unzip -o '{name}'")

assert (pathlib.Path(ADAPTER) / "adapter_config.json").exists(), "адаптер не найден"
print("Адаптер на месте:", ADAPTER)

## 2. Слияние с base + экспорт GGUF Q4_K_M (~15–25 минут)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="outputs/adapter_e3_lr2e-4",
    max_seq_length=2048,
    load_in_4bit=True,
)
model.save_pretrained_gguf("gguf_out", tokenizer, quantization_method="q4_k_m")

import glob
gguf_files = glob.glob("gguf_out/**/*.gguf", recursive=True)
print("GGUF-файлы:", gguf_files)
assert gguf_files, "GGUF не создался"
GGUF_PATH = gguf_files[0]
print("Размер:", round(os.path.getsize(GGUF_PATH) / 1e9, 2), "ГБ")

## 3. Скачать GGUF на компьютер (~4.4 ГБ, файл уйдёт в Загрузки)

In [ ]:
from google.colab import files
files.download(GGUF_PATH)

## 3.1 (опционально) Запушить GGUF на HuggingFace Hub

Понадобится для деплоя на HF Spaces. Локальному Docker-деплою не нужен.
Перед запуском: Colab → Secrets → `HF_TOKEN` (Write).

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

REPO_ID = "ВАШ_НИК/qwen2.5-7b-tk-gguf"  # поменяй на свой
from huggingface_hub import HfApi
api = HfApi(token=os.environ.get("HF_TOKEN"))
api.create_repo(REPO_ID, exist_ok=True)
api.upload_file(path_or_fileobj=GGUF_PATH,
                path_in_repo=GGUF_PATH.split("/")[-1],
                repo_id=REPO_ID, repo_type="model")
print(f"Загружено: https://huggingface.co/{REPO_ID}")